In [12]:
!pip install qdrant_client numpy

# Vector store
Come vector store suggerisco Qdrant https://qdrant.tech/. Si possono salvare su Qdrant i vettori e un payload json che può contenere il chunk insieme ai metadati. Alternativamente si può usare Qdrant per gestire solo i vettori e i metadati e opensearch per salvare i testi. N.B. In tal caso i metadati dei vettori devono contenere l'id del chunk salvato su opensearch.

## Creare una collection

In [1]:
import qdrant_client
from qdrant_client import models
client = qdrant_client.AsyncQdrantClient('http://localhost:6333', timeout=1000)

Creiamo una collection con una dimensione dei vettori settata a 768

In [71]:
await client.create_collection(
            collection_name='test_collection',
            vectors_config={
                "dense": models.VectorParams(
                    size=768,
                    distance=models.Distance.COSINE
                )
            }
)

True

Controllare su http://localhost:6333/dashboard che sia comparsa la collection "test_collection"

## Aggiungere un punto con metadati alla collection 
Si possono anche aggiungere più punti contemporaneamente

In [72]:
import numpy as np
import uuid

In [73]:
vector_1 = np.random.normal(size = (768,))
vector_2 = np.random.normal(size=(768,))
vector_3 = np.random.normal(size = (768,))
vector_4 = np.random.normal(size=(768,))

In [74]:
document_1 = {
    "text": "Questo è il primo documento",
    "tags": ["tag_2"],
    "metadata": {
        "classe": "prima media",
        "argomento": "storia",
    } 
}

In [75]:
document_2 = {
    "text": "Questo è il secondo documento",
    "tags": ["tag_1"],
    "metadata": {
        "classe": "prima elementare",
        "argomento": "geografia",
    } 
}

In [76]:
document_3 = {
    "text": "Questo è il terzo documento",
    "tags": ["tag_1", "tag_2"],
    "metadata": {
        "classe": "prima elementare",
        "argomento": "geografia",
    } 
}

In [77]:
document_4 = {
    "text": "Questo è il quarto documento",
    "tags": ["tag_2"],
    "metadata": {
        "classe": "prima elementare",
        "argomento": "geografia",
    } 
}

In [78]:
points = [(document_1, vector_1), (document_2, vector_2), (document_3, vector_3), (document_4, vector_4)]

In [79]:
await client.upsert(
                'test_collection',
                points=[
                    models.PointStruct(
                        id=uuid.uuid4(),
                        vector={
                            "dense": vector,
                        },
                        payload=doc
                    )
                for doc, vector in points])

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

Controllare su http://localhost:6333/dashboard#/collections/test_collection che ci siano i due punti

## Ricercare i punti per vicinanza con il vettore del prompt

In [32]:
vector_prompt = np.random.normal(size = (768,))

In [80]:
result = await client.query_points(collection_name='test_collection',
                                                query = vector_prompt,
                                                limit=4,
                                                using='dense',
                                                )

In [81]:
print(result)

points=[ScoredPoint(id='3f5f7f3b-53f7-4461-bd53-5089ab5d99dc', version=1, score=0.048300635, payload={'text': 'Questo è il secondo documento', 'tags': ['tag_1'], 'metadata': {'classe': 'prima elementare', 'argomento': 'geografia'}}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='ee44ecea-a294-4fc6-a975-9eab3f120780', version=1, score=-0.006552413, payload={'text': 'Questo è il terzo documento', 'tags': ['tag_1', 'tag_2'], 'metadata': {'classe': 'prima elementare', 'argomento': 'geografia'}}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='1500bf02-ccc7-4170-ac58-7fcf4d914592', version=1, score=-0.019092023, payload={'text': 'Questo è il primo documento', 'tags': ['tag_2'], 'metadata': {'classe': 'prima media', 'argomento': 'storia'}}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='eb0ccf6a-1b20-48e5-96d2-7cdf4837de10', version=1, score=-0.031116117, payload={'text': 'Questo è il quarto documento', 'tags': ['tag_2'], 'metadata': {'classe

In [82]:
print('------------Text----------------')
print(result.points[0].payload['text'])
print('------------Metadata------------')
print(result.points[0].payload['metadata'])

------------Text----------------
Questo è il secondo documento
------------Metadata------------
{'classe': 'prima elementare', 'argomento': 'geografia'}


## Ricercare i punti per vicinanza con il vettore del prompt e con filtro sui metadati

https://qdrant.tech/documentation/search/filtering/

Singola condizione

In [83]:
result = await client.query_points(
    collection_name='test_collection',
    query = vector_prompt,
    limit=4,
    using='dense',
    query_filter=models.Filter(
        must=[
            models.FieldCondition(
                key="metadata.classe", match=models.MatchValue(value="prima media")
            ),
        ],
    )
)

In [84]:
print(result)

points=[ScoredPoint(id='1500bf02-ccc7-4170-ac58-7fcf4d914592', version=1, score=-0.019092023, payload={'text': 'Questo è il primo documento', 'tags': ['tag_2'], 'metadata': {'classe': 'prima media', 'argomento': 'storia'}}, vector=None, shard_key=None, order_value=None)]


Condizione in AND

In [89]:
result = await client.query_points(
    collection_name='test_collection',
    query = vector_prompt,
    limit=4,
    using='dense',
    query_filter=models.Filter(
        must=[
            models.FieldCondition(
                key="metadata.classe", match=models.MatchValue(value="prima elementare")
            ),
            models.FieldCondition(
                key="metadata.argomento", match=models.MatchValue(value="geografia")
            ),
        ],
    )
)

In [90]:
print(result)

points=[ScoredPoint(id='3f5f7f3b-53f7-4461-bd53-5089ab5d99dc', version=1, score=0.048300635, payload={'text': 'Questo è il secondo documento', 'tags': ['tag_1'], 'metadata': {'classe': 'prima elementare', 'argomento': 'geografia'}}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='ee44ecea-a294-4fc6-a975-9eab3f120780', version=1, score=-0.006552413, payload={'text': 'Questo è il terzo documento', 'tags': ['tag_1', 'tag_2'], 'metadata': {'classe': 'prima elementare', 'argomento': 'geografia'}}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='eb0ccf6a-1b20-48e5-96d2-7cdf4837de10', version=1, score=-0.031116117, payload={'text': 'Questo è il quarto documento', 'tags': ['tag_2'], 'metadata': {'classe': 'prima elementare', 'argomento': 'geografia'}}, vector=None, shard_key=None, order_value=None)]


Condizione in OR

In [93]:
result = await client.query_points(
    collection_name='test_collection',
    query = vector_prompt,
    limit=4,
    using='dense',
    query_filter=models.Filter(
        should=[ #QUI ABBIAMO MESSO SHOULD
            models.FieldCondition(
                key="metadata.classe", match=models.MatchValue(value="prima media")
            ),
            models.FieldCondition(
                key="metadata.classe", match=models.MatchValue(value="prima elementare")
            ),
        ],
    )
)

In [94]:
print(result)

points=[ScoredPoint(id='3f5f7f3b-53f7-4461-bd53-5089ab5d99dc', version=1, score=0.048300635, payload={'text': 'Questo è il secondo documento', 'tags': ['tag_1'], 'metadata': {'classe': 'prima elementare', 'argomento': 'geografia'}}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='ee44ecea-a294-4fc6-a975-9eab3f120780', version=1, score=-0.006552413, payload={'text': 'Questo è il terzo documento', 'tags': ['tag_1', 'tag_2'], 'metadata': {'classe': 'prima elementare', 'argomento': 'geografia'}}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='1500bf02-ccc7-4170-ac58-7fcf4d914592', version=1, score=-0.019092023, payload={'text': 'Questo è il primo documento', 'tags': ['tag_2'], 'metadata': {'classe': 'prima media', 'argomento': 'storia'}}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='eb0ccf6a-1b20-48e5-96d2-7cdf4837de10', version=1, score=-0.031116117, payload={'text': 'Questo è il quarto documento', 'tags': ['tag_2'], 'metadata': {'classe

Condizione su liste

In [101]:
result_1 = await client.query_points(
    collection_name='test_collection',
    query = vector_prompt,
    limit=4,
    using='dense',
    query_filter=models.Filter(
        must=[ 
            models.FieldCondition(
                key="tags[]", match=models.MatchValue(value="tag_1")
            ),
            models.FieldCondition(
                key="tags[]", match=models.MatchValue(value="tag_2")
            ),
        ],
    )
)

result_2 =  await client.query_points(
    collection_name='test_collection',
    query = vector_prompt,
    limit=4,
    using='dense',
    query_filter=models.Filter(
        should=[ #QUI ABBIAMO MESSO SHOULD
            models.FieldCondition(
                key="tags[]", match=models.MatchValue(value="tag_1")
            ),
            models.FieldCondition(
                key="tags[]", match=models.MatchValue(value="tag_2")
            ),
        ],
    )
)

In [100]:
print(result_1)
print('-'*150)
print(result_2)

points=[ScoredPoint(id='ee44ecea-a294-4fc6-a975-9eab3f120780', version=1, score=-0.006552413, payload={'text': 'Questo è il terzo documento', 'tags': ['tag_1', 'tag_2'], 'metadata': {'classe': 'prima elementare', 'argomento': 'geografia'}}, vector=None, shard_key=None, order_value=None)]
------------------------------------------------------------------------------------------------------------------------------------------------------
points=[ScoredPoint(id='3f5f7f3b-53f7-4461-bd53-5089ab5d99dc', version=1, score=0.048300635, payload={'text': 'Questo è il secondo documento', 'tags': ['tag_1'], 'metadata': {'classe': 'prima elementare', 'argomento': 'geografia'}}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='ee44ecea-a294-4fc6-a975-9eab3f120780', version=1, score=-0.006552413, payload={'text': 'Questo è il terzo documento', 'tags': ['tag_1', 'tag_2'], 'metadata': {'classe': 'prima elementare', 'argomento': 'geografia'}}, vector=None, shard_key=None, order_value=None)

## Creare una collection per la ricerca ibrida

In [ ]:
import qdrant_client
from qdrant_client import models
client = qdrant_client.AsyncQdrantClient('http://localhost:6333', timeout=1000)

In [23]:
await client.create_collection(
            collection_name='hybrid_collection',
            vectors_config={
                "dense": models.VectorParams(
                    size=384,
                    distance=models.Distance.COSINE
                )
            },
    sparse_vectors_config={
        "bm25": models.SparseVectorParams(modifier=models.Modifier.IDF)
    }
)


True

## Inserimento dei documenti. Iniziamo ad usare Fastembed per calcolare gli embeddings

In [104]:
!pip install fastembed

  Using cached loguru-0.7.3-py3-none-any.whl.metadata (22 kB)
  Using cached py_rust_stemmers-0.1.5-cp311-none-win_amd64.whl.metadata (3.5 kB)
  Using cached win32_setctime-1.2.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-14.3.3-py3-none-any.whl.metadata (18 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/618.0 kB ? eta -:--:--
   ---------------------------------------- 618.0/618.0 kB 11.5 MB/s  0:00:00
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 3.7/3.7 MB 21.9 MB/s  0:00:00
Using cached loguru-0.7.3-py3-none-any.whl (61 kB)
   ---------------------------------------- 

In [68]:
from fastembed import TextEmbedding, SparseTextEmbedding, LateInteractionTextEmbedding

dense_embedding_model = TextEmbedding("sentence-transformers/all-MiniLM-L6-v2", cache_dir = './fastembed/')
bm25_embedding_model = SparseTextEmbedding("Qdrant/bm25", cache_dir = './fastembed/')
late_interaction_embedding_model = LateInteractionTextEmbedding("colbert-ir/colbertv2.0")



Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

C:\Users\CT-01\PycharmProjects\rag\venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\CT-01\AppData\Local\Temp\fastembed_cache\models--colbert-ir--colbertv2.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [8]:
documents = [
    "Parlare di Dante Alighieri significa confrontarsi non solo con il 'Sommo Poeta', ma con l'architetto stesso della lingua italiana. Nato a Firenze nel 1265, Dante ha attraversato un’epoca di feroci conflitti politici, pagando con l’esilio il suo impegno pubblico, ma trasformando quella sofferenza nel più grande viaggio letterario dell'umanità. Il cuore della sua eredità è la Divina Commedia, un’opera mondo che sintetizza la cultura medievale proiettandola verso l’umanesimo. Attraverso Inferno, Purgatorio e Paradiso, compie un percorso di redenzione universale. Dante resta attuale come padre della lingua italiana, per il suo fermo impegno politico contro la corruzione e per la profondità psicologica dei suoi personaggi. Infine, la figura di Beatrice eleva il concetto di amore verso una dimensione spirituale e salvifica. In sintesi, Dante ha racchiuso l’intera esperienza umana in versi perfetti, lasciandoci una mappa morale che continua a interrogarci su cosa significhi, davvero, essere uomini.",
    "I ragni sono artropodi predatori appartenenti all'ordine degli Araneae, distinti dagli insetti per avere otto zampe, il corpo diviso in due segmenti (cefalotorace e addome) e l'assenza di ali o antenne. Sono celebri per la loro capacità di produrre seta, una fibra proteica sottile ma incredibilmente resistente, utilizzata non solo per tessere ragnatele destinate alla cattura delle prede, ma anche per costruire sacchi ovigeri o come filo di sicurezza. Quasi tutte le specie possiedono ghiandole velenifere collegate ai cheliceri, che utilizzano per immobilizzare le prede, sebbene solo una minima parte sia pericolosa per l'essere umano. Svolgono un ruolo ecologico fondamentale come regolatori delle popolazioni di insetti, agendo come naturali controllori di parassiti in quasi ogni ecosistema terrestre. Nonostante la diffusa aracnofobia, i ragni mostrano comportamenti complessi, dalle elaborate danze di corteggiamento alle strategie di caccia attiva senza l'uso di tele, confermandosi tra le creature più evolute e adattabili del pianeta.",
    "La pesca sportiva è un'attività ricreativa e agonistica che si distingue dalla pesca commerciale per l'assenza di fini di lucro e per l'enfasi posta sulla sfida tra pescatore e preda. Praticata in acque dolci (fiumi, laghi) o in mare, si avvale di attrezzature specifiche come canne, mulinelli, lenze e ami, adattati a diverse tecniche: dallo spinning alla pesca a mosca, fino al surfcasting e alla passata. Un pilastro moderno di questa disciplina è il 'Catch and Release' (cattura e rilascio), una pratica etica che prevede la liberazione del pesce dopo la cattura per preservare la biodiversità e garantire la sostenibilità degli stock ittici. Oltre all'aspetto tecnico, la pesca sportiva richiede una profonda conoscenza dell'etologia delle specie e dei cicli stagionali, trasformandosi in un esercizio di pazienza e osservazione che favorisce il benessere psicofisico e la connessione con la natura. Le competizioni ufficiali, regolate da federazioni internazionali, premiano la precisione e l'abilità strategica, elevando questo hobby a vero e proprio sport di destrezza.",
    "La pesca è il frutto della pianta Prunus persica, un albero appartenente alla famiglia delle Rosaceae originario della Cina, dove è considerato simbolo di immortalità e giovinezza. Caratterizzata da una buccia vellutata (nella varietà comune) o liscia (nelle nettarine o pesche noci), la sua polpa può variare dal bianco al giallo intenso, fino a sfumature rossastre vicino al nocciolo legnoso. Dal punto di vista nutrizionale, la pesca è un alimento ipocalorico ma ricco di proprietà benefiche: è composta per circa il 90% da acqua, il che la rende altamente rinfrescante, ed è un'ottima fonte di vitamina A, vitamina C e potassio. Oltre al consumo fresco, la pesca è estremamente versatile in cucina, venendo impiegata per la preparazione di confetture, succhi, sciroppi e dessert tradizionali. La stagionalità del frutto, che raggiunge il picco di maturazione tra giugno e settembre, lo rende l'emblema della dieta mediterranea estiva, apprezzato non solo per la sua dolcezza equilibrata da una lieve acidità, ma anche per le sue doti antiossidanti e depurative.",
    "Leonardo S.p.A. è un'azienda globale italiana ad alta tecnologia, tra i primi dieci operatori mondiali nel settore dell'Aerospazio, Difesa e Sicurezza. Con sede a Roma, la società è guidata da una forte impronta industriale focalizzata su quattro divisioni principali: Elicotteri, Velivoli, Elettronica per la Difesa e Sicurezza e Spazio. Leonardo progetta e realizza soluzioni all'avanguardia come i convertiplani, i velivoli da addestramento avanzato (come l'M-346) e sistemi radar complessi, integrando intelligenza artificiale e cybersecurity nelle proprie piattaforme. Partecipa inoltre a programmi internazionali strategici, inclusa la gestione di costellazioni satellitari attraverso joint venture come Thales Alenia Space e Telespazio. Con oltre 50.000 dipendenti e una presenza capillare in mercati chiave come Regno Unito, USA e Polonia, l'azienda rappresenta un asset strategico per la sovranità tecnologica europea, investendo costantemente in ricerca e sviluppo per guidare la transizione verso la digitalizzazione del campo di battaglia e la mobilità aerea del futuro.",
    "Identificato per la prima volta nel 2007, Zeus è un Trojan horse progettato principalmente per sottrarre informazioni finanziarie attraverso tecniche di 'man-in-the-browser', keylogging e form grabbing. La sua pericolosità risiede nella capacità di intercettare le credenziali bancarie mentre l'utente le digita su siti legittimi, bypassando spesso l'autenticazione a due fattori. Nel 2011, il leak del suo codice sorgente ha dato vita a innumerevoli varianti, tra cui il celebre 'Gameover ZeuS', che ha introdotto una struttura peer-to-peer (P2P) decentralizzata rendendo estremamente difficile il takedown da parte delle autorità. Oltre al furto di dati, Zeus è stato utilizzato come veicolo per la diffusione di ransomware come CryptoLocker, causando danni stimati in centinaia di milioni di dollari a livello globale. Nonostante le operazioni internazionali di contrasto (come l'Operazione Tovar del 2014), l'eredità di Zeus sopravvive oggi nel codice di molti moderni malware finanziari, confermandosi come il precursore delle moderne botnet professionali e del modello 'Cybercrime-as-a-Service'.",
    "Nella mitologia greca, Zeus è il re degli dèi, il sovrano dell'Olimpo e il signore del cielo, dei tuoni e dei fulmini. Figlio dei titani Crono e Rea, Zeus ascese al potere dopo aver guidato i propri fratelli nella Titanomachia, rovesciando la tirannia del padre che usava divorare i propri figli. È descritto come una figura maestosa e autoritaria, il cui simbolo principale è la folgore, forgiata per lui dai Ciclopi. Oltre a garantire l'ordine cosmico e la giustizia, Zeus è noto per la sua complessa vita sentimentale e le innumerevoli metamorfosi (come il cigno, il toro o la pioggia d'oro) adottate per unirsi a dee e mortali, dando origine a una vasta stirpe di eroi e divinità tra cui Eracle, Atena e Apollo. Nonostante la sua saggezza, il suo carattere è spesso impulsivo e vendicativo, incarnando la forza imprevedibile degli elementi naturali. Il suo culto era centrale nell'antica Grecia, con il santuario di Olimpia che ospitava i giochi a lui dedicati e una delle Sette Meraviglie del mondo antico: la sua colossale statua crisoelefantina realizzata da Fidia.",
    "Situata all'estremità meridionale della penisola balcanica, la Grecia funge da ponte naturale tra Europa, Asia e Africa. Il suo territorio è caratterizzato da una morfologia complessa: circa l'80% della superficie è montuoso, con il Monte Olimpo che svetta come cima più alta (2.918 m). La particolarità geografica più rilevante è l'estensione delle sue coste, che superano i 13.600 km grazie a una frammentazione estrema e alla presenza di oltre 2.000 isole, raggruppate in arcipelaghi storici come le Cicladi, il Dodecaneso, le Ionie e le Sporadi, oltre alla grande isola di Creta. Il clima è tipicamente mediterraneo lungo le coste, con estati calde e secche, mentre diventa continentale con inverni rigidi nelle regioni interne del Nord (Macedonia ed Epiro). Dal punto di vista idrografico, i fiumi sono brevi e a regime torrentizio, riflettendo l'aridità di molte zone. Questa conformazione geografica, fatta di valli isolate e accessi diretti al mare, ha storicamente favorito lo sviluppo delle città-stato indipendenti e la vocazione marittima che ancora oggi definisce l'economia e l'identità del Paese.",
    "Giorgia Meloni (Roma, 1977) è una politica e giornalista italiana, Presidente del Consiglio dei Ministri dal 22 ottobre 2022, prima donna a ricoprire tale incarico nella storia della Repubblica. La sua carriera politica inizia a 15 anni nel Fronte della Gioventù; successivamente guida Azione Giovani e nel 2008 diventa il più giovane Ministro della Gioventù nel governo Berlusconi IV. Nel 2012 fonda Fratelli d'Italia, partito di cui è presidente e che guida a una crescita costante fino alla vittoria nelle elezioni politiche del 2022. A livello internazionale, ricopre la presidenza del Partito dei Conservatori e Riformisti Europei (ECR Party) e, nel 2024, ha presieduto il G7 a guida italiana. Nel marzo 2026, il suo governo è impegnato su riforme istituzionali chiave, tra cui il referendum sulla giustizia per la separazione delle carriere e l'attuazione del PNRR. La sua linea politica si caratterizza per il conservatorismo sociale, il sovranismo economico temperato da un forte atlantismo e una costante attenzione alla difesa dell'identità nazionale e della famiglia.",
    "Alessandro Manzoni (Milano, 1785-1873) è stato uno scrittore, poeta e drammaturgo, la cui opera ha segnato profondamente l'identità culturale e linguistica dell'Italia unita. Cresciuto tra l'Illuminismo parigino e la fede cattolica ritrovata (la 'Conversione' del 1810), Manzoni ha saputo coniugare il rigore morale con un profondo realismo storico. Il suo capolavoro, 'I Promessi Sposi', rappresenta il primo vero romanzo moderno in lingua italiana: attraverso la vicenda dei due umili tessitori, Renzo e Lucia, Manzoni non solo critica l'oppressione straniera e le ingiustizie sociali, ma compie una rivoluzione linguistica abbattendo la barriera tra lingua letteraria e lingua parlata ('sciacquando i panni in Arno'). Oltre al romanzo, la sua produzione spazia dalle odi civili (come il 'Cinque Maggio' dedicato a Napoleone) agli Inni Sacri e alle tragedie (Adelchi e Il Conte di Carmagnola), in cui introduce il concetto di 'vero storico' unito al 'verosimile'. Senatore del Regno d'Italia, Manzoni rimane oggi il simbolo di un intellettuale civile che ha saputo dare agli italiani una lingua comune e una coscienza nazionale fondata sulla Provvidenza e sulla dignità umana.",
    "La Serie A è il vertice del sistema calcistico italiano e una delle leghe più prestigiose al mondo, nota storicamente per il suo rigore tattico e la qualità dei suoi difensori. Fondata nel 1898, la competizione ha assunto la formula del 'girone unico' nella stagione 1929-1930. Oggi il campionato vede la partecipazione di 20 squadre che si contendono lo Scudetto e il titolo di Campione d'Italia, simboleggiato dal tricolore apposto sulle maglie della squadra vincitrice nella stagione successiva. La Juventus detiene il record di titoli vinti, seguita dalle milanesi Inter e Milan, che nel 2024 hanno celebrato rispettivamente la conquista della seconda stella. La Serie A non è solo un evento sportivo, ma un asset economico e culturale fondamentale per l'Italia, capace di attrarre campioni internazionali e generare un vasto indotto mediatico. Negli ultimi anni, il campionato ha vissuto una fase di rilancio internazionale grazie a una maggiore propensione allo spettacolo offensivo e ai successi dei club italiani nelle competizioni europee, mantenendo il fascino di 'campionato più difficile al mondo' per la preparazione atletica e strategica richiesta.",
    "L'Intelligenza Artificiale (IA) nel 2026 ha superato la fase dell'entusiasmo mediatico per diventare una componente infrastrutturale dell'economia globale. La tendenza dominante è l'avvento dell'AI Agentica: sistemi non più solo conversazionali, ma capaci di pianificare ed eseguire flussi di lavoro complessi in autonomia (Agentic AI), agendo come veri e propri colleghi digitali. Parallelamente, la 'Physical AI' sta integrando modelli neurali avanzati nella robotica e nell'IoT, permettendo alle macchine di interagire fisicamente con l'ambiente in tempo reale. In Italia, il mercato ha raggiunto gli 1,8 miliardi di euro, con una forte spinta verso l'industrializzazione dei processi aziendali e l'adozione diffusa della Generative Engine Optimization (GEO). Tuttavia, l'espansione tecnologica ha sollevato nuove sfide critiche: dalla cybersecurity, con attacchi di phishing automatizzati in grado di profilare vittime in pochi minuti, alla necessità di certificare l'autenticità dei contenuti tramite watermark crittografici (Digital Provenance). Mentre l'Europa implementa le linee guida dell'AI Act, il focus si è spostato dal puro potenziale tecnologico al ROI concreto e alla sostenibilità energetica dei grandi data center, ridefinendo il mercato del lavoro attraverso una richiesta massiccia di competenze ibride tra tecnica e gestione dei processi.",
    "Il Machine Learning (ML) è una branca dell'intelligenza artificiale che si occupa di creare sistemi capaci di apprendere dai dati, migliorando le proprie prestazioni nel tempo senza essere esplicitamente programmati per ogni singola attività. A differenza del software tradizionale basato su regole rigide (if-then), il ML utilizza algoritmi statistici per identificare pattern complessi in enormi volumi di informazioni. Nel 2026, il campo si è evoluto oltre il deep learning classico verso architetture più efficienti come i modelli basati sullo 'State Space' e il 'Few-shot Learning', che permettono alle macchine di imparare concetti nuovi con pochissimi esempi. Le tre categorie fondamentali rimangono l'apprendimento supervisionato (predizioni basate su dati etichettati), non supervisionato (scoperta di strutture nascoste) e per rinforzo (ottimizzazione tramite tentativi ed errori). Oggi, il Machine Learning è integrato in ogni aspetto della vita digitale: dalla diagnostica medica predittiva alla manutenzione industriale 4.0, fino alla personalizzazione estrema dell'esperienza utente, rendendo i sistemi non solo reattivi, ma capaci di anticipare le necessità umane.",
    "Umberto Bossi (1941-2026), fondatore della Lega Nord e figura iconica della politica italiana, si è spento il 19 marzo 2026 a Varese all'età di 84 anni. Noto come il 'Senatùr', Bossi ha trasformato la protesta settentrionale contro 'Roma ladrona' in un movimento politico di massa, portando per la prima volta le istanze del federalismo e della devoluzione al centro del dibattito nazionale. Sotto la sua guida, la Lega Lombarda (poi Lega Nord) divenne l'ago della bilancia dei governi di centrodestra degli anni '90 e 2000. Dopo l'ictus del 2004 e le dimissioni da segretario nel 2012, Bossi ha ricoperto il ruolo di Presidente a vita, mantenendo un atteggiamento critico verso la svolta nazionalista impressa da Matteo Salvini con la nascita della 'Lega per Salvini Premier'. Nel marzo 2026, la sua scomparsa segna la fine di un'era proprio mentre il partito discute un ritorno alle origini autonomiste e la possibile rimozione del nome del leader dal simbolo in vista del 2027. L'eredità di Bossi rimane impressa nella trasformazione dell'Italia in senso regionale e nella nascita di un'identità politica territoriale che, pur evolvendosi, continua a influenzare gli equilibri della coalizione di governo.",
    "Il conflitto in Iran, entrato in una fase di escalation aperta nel marzo 2026, rappresenta lo scontro diretto tra la Repubblica Islamica e l'asse composto da Israele e Stati Uniti (operazioni 'Epic Fury' e 'Roaring Lion'). La crisi, scaturita dal fallimento dei negoziati sul nucleare e da una serie di raid israeliani del 2025, è degenerata il 28 febbraio 2026 con un massiccio attacco su Teheran che ha portato alla morte della Guida Suprema Ali Khamenei, a cui è succeduto il figlio Mojtaba. Al 20 marzo 2026, il conflitto si concentra sul controllo dello Stretto di Hormuz, chiuso dai Pasdaran con conseguenze devastanti sui mercati energetici globali e sull'inflazione in Europa. Israele continua un'ondata di raid strategici su infrastrutture militari e petrolifere (inclusi siti sul Mar Caspio), mentre gli USA valutano l'occupazione dell'isola di Kharg per ripristinare il flusso di greggio. Sul fronte interno, il regime affronta una grave crisi di stabilità, esacerbata dall'uccisione di figure chiave come il portavoce delle IRGC, Ali Mohammad Naeini. L'Italia, per voce del Premier Meloni, ha ribadito la propria linea di non intervento militare diretto, pur sostenendo gli sforzi diplomatici per evitare una crisi energetica senza precedenti nella storia moderna.",
    "Immanuel Kant (1724-1804) è stato uno dei più influenti filosofi dell'Illuminismo, noto per aver operato una sintesi definitiva tra razionalismo ed empirismo. La sua opera si fonda sul criticismo, un'indagine sui limiti e le possibilità della ragione umana espressa in tre capolavori: la 'Critica della ragion pura' (conoscenza), la 'Critica della ragion pratica' (etica) e la 'Critica del Giudizio' (estetica). Kant ha introdotto l'idea che non è la mente a doversi adattare agli oggetti, ma sono gli oggetti a essere modellati dalle nostre strutture conoscitive (spazio, tempo e categorie). In campo morale, ha formulato l'imperativo categorico, un principio universale che impone di agire secondo massime che possano valere per tutti, ponendo la libertà e la dignità umana al centro dell'agire. Il suo pensiero ha gettato le basi per la filosofia moderna, influenzando profondamente il diritto, la politica e la scienza, e promuovendo l'ideale di una pace perpetua fondata sul cosmopolitismo e sul rispetto delle leggi internazionali."
]

In [9]:
dense_embeddings = list(dense_embedding_model.embed(doc for doc in documents))
bm25_embeddings = list(bm25_embedding_model.embed(doc for doc in documents))

In [25]:
dense_embeddings[0][:10]

array([-0.03169013,  0.05281247, -0.06523138,  0.03500904, -0.05331872,
        0.06014788,  0.05868977,  0.03280578, -0.01400479, -0.06279175])

In [14]:
bm25_embeddings[0] #Poiché FastEmbed usa un metodo chiamato Feature Hashing, questi numeri sono molto grandi e servono a mappare le parole in uno spazio virtuale immenso (miliardi di possibili "posizioni") senza dover gestire un file di dizionario gigante sul disco.

SparseEmbedding(values=array([1.21327014, 1.89221879, 1.82827463, 1.21327014, 1.21327014,
       1.21327014, 1.21327014, 1.21327014, 1.73079287, 1.89221879,
       1.21327014, 1.21327014, 1.82827463, 1.21327014, 1.21327014,
       1.73079287, 1.56401   , 1.56401   , 1.21327014, 1.21327014,
       1.56401   , 1.21327014, 1.56401   , 1.21327014, 1.73079287,
       1.21327014, 1.21327014, 1.21327014, 1.21327014, 1.21327014,
       1.21327014, 1.56401   , 1.56401   , 1.21327014, 1.21327014,
       1.21327014, 1.21327014, 1.21327014, 1.21327014, 1.21327014,
       1.21327014, 1.21327014, 1.21327014, 1.21327014, 1.21327014,
       1.21327014, 1.21327014, 1.89221879, 1.21327014, 1.21327014,
       1.21327014, 1.21327014, 1.56401   , 1.21327014, 1.21327014,
       1.21327014, 1.21327014, 1.56401   , 1.21327014, 1.21327014,
       1.21327014, 1.21327014, 1.73079287, 1.21327014, 1.21327014,
       1.21327014, 1.21327014, 1.21327014, 1.21327014, 1.21327014,
       1.21327014, 1.21327014, 1.56401 

In [17]:
from qdrant_client.models import PointStruct

points = []
for idx, (dense_embedding, bm25_embedding, doc) in enumerate(zip(dense_embeddings, bm25_embeddings, documents)):
  
    point = PointStruct(
        id=idx,
        vector={
            "dense": dense_embedding,
            "bm25": bm25_embedding.as_object(),
        },
        payload={"document": doc}
    )
    points.append(point)

In [24]:
operation_info = await client.upsert(
    collection_name="hybrid_collection",
    points=points
)
print(operation_info)

operation_id=1 status=<UpdateStatus.COMPLETED: 'completed'>


C:\Users\CT-01\AppData\Local\Temp\ipykernel_20184\197245143.py:1: RuntimeWarning: coroutine 'AsyncQdrantClient.upsert' was never awaited
  operation_info = await client.upsert(


## Cerchiamo un documento usando solo il vettore denso

In [46]:
query = "Dammi un esempio di CryptoLocker"

In [47]:
dense_vectors = next(dense_embedding_model.query_embed(query))
sparse_vectors = next(bm25_embedding_model.query_embed(query))


In [51]:
result = await client.query_points(collection_name='hybrid_collection',
                                                query = dense_vectors,
                                                limit=1,
                                                using='dense',
                                                )
result

QueryResponse(points=[ScoredPoint(id=1, version=1, score=0.457677, payload={'document': "I ragni sono artropodi predatori appartenenti all'ordine degli Araneae, distinti dagli insetti per avere otto zampe, il corpo diviso in due segmenti (cefalotorace e addome) e l'assenza di ali o antenne. Sono celebri per la loro capacità di produrre seta, una fibra proteica sottile ma incredibilmente resistente, utilizzata non solo per tessere ragnatele destinate alla cattura delle prede, ma anche per costruire sacchi ovigeri o come filo di sicurezza. Quasi tutte le specie possiedono ghiandole velenifere collegate ai cheliceri, che utilizzano per immobilizzare le prede, sebbene solo una minima parte sia pericolosa per l'essere umano. Svolgono un ruolo ecologico fondamentale come regolatori delle popolazioni di insetti, agendo come naturali controllori di parassiti in quasi ogni ecosistema terrestre. Nonostante la diffusa aracnofobia, i ragni mostrano comportamenti complessi, dalle elaborate danze di

## Cerchiamo solo con la ricerca sparsa

In [60]:
result = await client.query_points(collection_name='hybrid_collection',
                                                query = models.SparseVector(**sparse_vectors.as_object()),
                                                limit=2,
                                                using='bm25',
                                                )
result

QueryResponse(points=[ScoredPoint(id=5, version=1, score=3.3217025, payload={'document': "Identificato per la prima volta nel 2007, Zeus è un Trojan horse progettato principalmente per sottrarre informazioni finanziarie attraverso tecniche di 'man-in-the-browser', keylogging e form grabbing. La sua pericolosità risiede nella capacità di intercettare le credenziali bancarie mentre l'utente le digita su siti legittimi, bypassando spesso l'autenticazione a due fattori. Nel 2011, il leak del suo codice sorgente ha dato vita a innumerevoli varianti, tra cui il celebre 'Gameover ZeuS', che ha introdotto una struttura peer-to-peer (P2P) decentralizzata rendendo estremamente difficile il takedown da parte delle autorità. Oltre al furto di dati, Zeus è stato utilizzato come veicolo per la diffusione di ransomware come CryptoLocker, causando danni stimati in centinaia di milioni di dollari a livello globale. Nonostante le operazioni internazionali di contrasto (come l'Operazione Tovar del 2014),

## Abilitiamo la ricerca ibrida

In [52]:
prefetch = [
        models.Prefetch(
            query=dense_vectors,
            using="dense",
            limit=20,
        ),
        models.Prefetch(
            query=models.SparseVector(**sparse_vectors.as_object()),
            using="bm25",
            limit=20,
        ),
    ]

In [61]:
result = await client.query_points(collection_name='hybrid_collection',
                                   prefetch= prefetch,
                                                query=models.FusionQuery(fusion=models.Fusion.RRF),
                                                limit=2,
                                                )
result

QueryResponse(points=[ScoredPoint(id=5, version=1, score=0.8333334, payload={'document': "Identificato per la prima volta nel 2007, Zeus è un Trojan horse progettato principalmente per sottrarre informazioni finanziarie attraverso tecniche di 'man-in-the-browser', keylogging e form grabbing. La sua pericolosità risiede nella capacità di intercettare le credenziali bancarie mentre l'utente le digita su siti legittimi, bypassando spesso l'autenticazione a due fattori. Nel 2011, il leak del suo codice sorgente ha dato vita a innumerevoli varianti, tra cui il celebre 'Gameover ZeuS', che ha introdotto una struttura peer-to-peer (P2P) decentralizzata rendendo estremamente difficile il takedown da parte delle autorità. Oltre al furto di dati, Zeus è stato utilizzato come veicolo per la diffusione di ransomware come CryptoLocker, causando danni stimati in centinaia di milioni di dollari a livello globale. Nonostante le operazioni internazionali di contrasto (come l'Operazione Tovar del 2014),

## Adesso inseriamo il reranking con Colbert

In [67]:
collection_name = "rerankin_with_colbert"
await client.create_collection(
    collection_name=collection_name,
    vectors_config={
        "dense": models.VectorParams(
            size=384,
            distance=models.Distance.COSINE
        ),
        "colbert": models.VectorParams(
            size=128,
            distance=models.Distance.COSINE,
            multivector_config=models.MultiVectorConfig(
                comparator=models.MultiVectorComparator.MAX_SIM
            ),
            hnsw_config=models.HnswConfigDiff(m=0)  # Disable HNSW for reranking
        )
    },
    sparse_vectors_config={
        "bm25": models.SparseVectorParams(modifier=models.Modifier.IDF)
    }
)

True

In [69]:
late_interaction_embeddings = list(late_interaction_embedding_model.embed(doc for doc in documents))

In [75]:
late_interaction_embeddings[3].shape

(331, 128)

In [77]:
from qdrant_client.models import PointStruct
points = []
for idx, (dense_embedding, bm25_embedding, late_interaction_embedding, doc) in enumerate(zip(dense_embeddings, bm25_embeddings, late_interaction_embeddings, documents)):
  
    point = PointStruct(
        id=idx,
        vector={
            "dense": dense_embedding,
            "bm25": bm25_embedding.as_object(),
            "colbert": late_interaction_embedding,
        },
        payload={"document": doc}
    )
    points.append(point)

operation_info = await client.upsert(
    collection_name="rerankin_with_colbert",
    points=points
)

# Ricerca documenti

In [78]:
late_vectors = next(late_interaction_embedding_model.query_embed(query))

In [83]:
prefetch = [
        models.Prefetch(
            query=dense_vectors,
            using="dense",
            limit=20,
        ),
        models.Prefetch(
            query=models.SparseVector(**sparse_vectors.as_object()),
            using="bm25",
            limit=20,
        ),
    ]

results = await client.query_points(
         "rerankin_with_colbert",
        prefetch=prefetch,
        query=late_vectors,
        using="colbert",
        with_payload=True,
        limit=1,
)

C:\Users\CT-01\AppData\Local\Temp\ipykernel_20184\3415485607.py:14: RuntimeWarning: coroutine 'AsyncQdrantClient.query_points' was never awaited
  results = await client.query_points(


In [84]:
results

QueryResponse(points=[ScoredPoint(id=5, version=1, score=16.596855, payload={'document': "Identificato per la prima volta nel 2007, Zeus è un Trojan horse progettato principalmente per sottrarre informazioni finanziarie attraverso tecniche di 'man-in-the-browser', keylogging e form grabbing. La sua pericolosità risiede nella capacità di intercettare le credenziali bancarie mentre l'utente le digita su siti legittimi, bypassando spesso l'autenticazione a due fattori. Nel 2011, il leak del suo codice sorgente ha dato vita a innumerevoli varianti, tra cui il celebre 'Gameover ZeuS', che ha introdotto una struttura peer-to-peer (P2P) decentralizzata rendendo estremamente difficile il takedown da parte delle autorità. Oltre al furto di dati, Zeus è stato utilizzato come veicolo per la diffusione di ransomware come CryptoLocker, causando danni stimati in centinaia di milioni di dollari a livello globale. Nonostante le operazioni internazionali di contrasto (come l'Operazione Tovar del 2014),